# 1. Data Generation — Qwen3-8B Teacher Inference

Use Qwen3-8B via vLLM to generate correct arithmetic expressions for Countdown puzzles.
These expressions become the training labels for Gemma-3-1B.

In [ ]:
!pip install "numpy<2" torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -q
!pip install vllm transformers accelerate datasets pandas tqdm -q

In [ ]:
import torch
print(f"Torch:          {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Load dataset

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("HuggingFaceTB/Countdown-Task-GOLD", "all")
df = dataset["train"].to_pandas()
print(f"Dataset size: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

## Load Qwen3-8B via vLLM

In [ ]:
from vllm import LLM, SamplingParams

model_name = "Qwen/Qwen3-8B"

llm = LLM(
    model=model_name,
    trust_remote_code=True,
    dtype="float16",
    tensor_parallel_size=1,
    enforce_eager=True,
    max_model_len=4096,
)

sampling_params = SamplingParams(
    temperature=0.1,
    top_p=0.9,
    max_tokens=256,
    stop=["<|im_end|>", "<|endoftext|>"],
)

print(f"Model {model_name} loaded")

## Define prompt and response parsing

In [ ]:
def create_prompts(df_batch):
    prompts = []
    for _, row in df_batch.iterrows():
        target = row['target']
        nums = row['nums']
        nums_str = str(nums.tolist()) if hasattr(nums, 'tolist') else str(nums)

        prompt = (
            f"<|im_start|>system\n"
            f"You are a math assistant that solves Countdown puzzles.<|im_end|>\n"
            f"<|im_start|>user\n"
            f"Given the target number {target} and the list of numbers {nums_str}, "
            f"find a mathematical expression using basic operations (+, -, *, /) that equals the target. "
            f"You can use each number exactly once.\n"
            f"Provide your answer as a mathematical expression only, no explanation.<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        prompts.append(prompt)
    return prompts


def parse_answer(response):
    """Extract the expression from model response."""
    text = response.strip()
    # Remove any leftover tags
    for tag in ["<|im_end|>", "<|endoftext|>"]:
        text = text.split(tag)[0].strip()
    return text

## Generate and save incrementally

In [ ]:
import os
from tqdm import tqdm

def generate_and_save(df, output_file, batch_size=100, max_samples=None):
    if max_samples and max_samples < len(df):
        df = df.head(max_samples).copy()

    total = len(df)
    all_results = []
    print(f"Processing {total} rows...")

    for start in tqdm(range(0, total, batch_size)):
        end = min(start + batch_size, total)
        batch = df.iloc[start:end]

        prompts = create_prompts(batch)
        outputs = llm.generate(prompts, sampling_params)

        batch_results = []
        for (_, row), out in zip(batch.iterrows(), outputs):
            answer = parse_answer(out.outputs[0].text)
            batch_results.append({
                "target": row["target"],
                "nums":   str(row["nums"]),
                "expression": answer,
            })

        all_results.extend(batch_results)
        chunk_df = pd.DataFrame(batch_results)
        chunk_df.to_csv(
            output_file,
            mode="w" if start == 0 else "a",
            header=(start == 0),
            index=False,
            encoding="utf-8",
        )
        print(f"  saved {end}/{total} rows to {output_file}")

    return pd.DataFrame(all_results)

## Run

In [ ]:
results_df = generate_and_save(
    df=df,
    output_file="qwen3_full_results.csv",
    batch_size=100,
    max_samples=None,   # set e.g. 500 to test quickly
)

print(f"Done. Total rows: {len(results_df)}")
print(results_df.head())